In [2]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


from scipy import stats
from statsmodels.stats.proportion import proportion_confint


con=sqlite3.connect('C:\\Users\\moham\\OneDrive\\Desktop\\SEM 3 OJT\\Proj_3.1\\data\\raw\\ipl.db')
def q(sql):
    return pd.read_sql(sql, con)

In [3]:
stats.binomtest(613, 1187, 0.5).pvalue

np.float64(0.2700389712415702)

In [4]:
q("""SELECT COUNT(*) AS n,
       SUM(CASE WHEN toss_winner = match_winner THEN 1 ELSE 0 END) AS k
FROM matches_clean
WHERE result = 'win';""")

,n,k
0,1046,541


In [5]:
# syntax :binomtest(k, n, p)
# k=no of success
# n=total trail,data
# p=expected probability
p = stats.binomtest(613, 1187, 0.5).pvalue
ci = proportion_confint(613, 1187, method='wilson')
print(p)
print(ci)

0.2700389712415702
(0.48799203833275506, 0.5447579131499739)


In [6]:
q(""" SELECT toss_decision,
    SUM(CASE WHEN toss_winner= match_winner
             THEN 1 ELSE 0 END) AS WON,
    SUM(CASE WHEN toss_winner <> match_winner
                 THEN 1 ELSE 0 END) AS lost
FROM matches_clean
WHERE result = 'win'
GROUP BY toss_decision;    





""")

,toss_decision,WON,lost
0,bat,164,194
1,field,377,311


In [7]:
q("""SELECT CASE WHEN first_innings_runs < 140 THEN 'UNDER 140'
            WHEN first_innings_runs < 160 THEN '140-159'
            WHEN first_innings_runs < 180 THEN '160-179'
             WHEN first_innings_runs < 200 THEN '180-199'
             ELSE '200+' END AS band,
        SUM(chase_won)       AS won,
        SUM(1 - chase_won)   AS lost 
FROM    v_match_totals
GROUP   BY band;       

""")

,band,won,lost
0,140-159,148,61
1,160-179,149,132
2,180-199,73,119
3,200+,36,131
4,UNDER 140,165,32


In [8]:
q("SELECT * FROM teams;")

,team_id,team_name
0,1,Royal Challengers Bangalore
1,2,Sunrisers Hyderabad
2,3,Mumbai Indians
3,4,Rising Pune Supergiant
4,5,Gujarat Lions
5,6,Kolkata Knight Riders
6,129,Chennai Super Kings
7,134,Rajasthan Royals
8,252,Delhi Capitals
9,494,Punjab Kings


In [9]:
q("""
SELECT *
FROM matches_clean
WHERE UPPER(toss_decision) = 'BAT';
""")

,match_id,venue_clean,city_clean,season_year,result,match_winner,player_of_match,toss_winner,toss_decision
0,1082595,M.Chinnaswamy Stadium,Bengaluru,2017,win,Royal Challengers Bangalore,KM Jadhav,Royal Challengers Bangalore,bat
1,1082598,Holkar Cricket Stadium,Indore,2017,win,Punjab Kings,AR Patel,Royal Challengers Bangalore,bat
2,1082605,Feroz Shah Kotla,Delhi,2017,win,Delhi Capitals,CJ Anderson,Delhi Capitals,bat
3,1082608,Feroz Shah Kotla,Delhi,2017,win,Kolkata Knight Riders,NM Coulter-Nile,Delhi Capitals,bat
4,1082611,Rajiv Gandhi International Stadium,Hyderabad,2017,win,Sunrisers Hyderabad,KS Williamson,Sunrisers Hyderabad,bat
...,...,...,...,...,...,...,...,...,...
363,1527685,Eden Gardens,Kolkata,2026,no result,Punjab Kings,,Kolkata Knight Riders,bat
364,1529271,Eden Gardens,Kolkata,2026,win,Kolkata Knight Riders,CV Varun,Rajasthan Royals,bat
365,1529278,Arun Jaitley Stadium,Delhi,2026,win,Punjab Kings,KL Rahul,Delhi Capitals,bat
366,1529284,Wankhede Stadium,Mumbai,2026,win,Sunrisers Hyderabad,H Klaasen,Mumbai Indians,bat


In [10]:
q("""
SELECT DISTINCT team_name
FROM teams
ORDER BY team_name;
""")

,team_name
0,Chennai Super Kings
1,Deccan Chargers
2,Delhi Capitals
3,Gujarat Lions
4,Gujarat Titans
5,Kochi Tuskers Kerala
6,Kolkata Knight Riders
7,Lucknow Super Giants
8,Mumbai Indians
9,Pune Warriors


In [11]:
q("SELECT COUNT(*) AS total_matches FROM matches;")

,total_matches
0,1212


In [12]:
q("""
SELECT season, COUNT(*) AS matches_played
FROM matches
GROUP BY season
ORDER BY season;
""")

,season,matches_played
0,2008,58
1,2009,57
2,2010,60
3,2011,73
4,2012,74
5,2013,76
6,2014,60
7,2015,59
8,2016,60
9,2017,59


In [13]:
q("""
SELECT match_winner AS team,
    COUNT(*) AS wins
FROM matches_clean
WHERE match_winner IS NOT NULL
GROUP BY match_winner
ORDER BY wins DESC
LIMIT 5;
""")

,team,wins
0,Mumbai Indians,146
1,Chennai Super Kings,138
2,Royal Challengers Bangalore,129
3,Kolkata Knight Riders,128
4,Sunrisers Hyderabad,123


In [14]:
#Are defended totals actually bigger than chased ones?

#h0=Both kinds of total average the same. The gap we see is luck
#h1=The two averages are not equal.

q("""SELECT first_innings_runs, chase_won
FROM   v_match_totals;""")

,first_innings_runs,chase_won
0,222,0
1,207,0
2,184,1
3,183,1
4,163,1
...,...,...
1041,228,1
1042,158,1
1043,75,1
1044,243,1


In [15]:
df=pd.read_sql("""SELECT first_innings_runs, chase_won
FROM   v_match_totals;""",con)
defended = df[df.chase_won == 0].first_innings_runs
chased   = df[df.chase_won == 1].first_innings_runs
t, p = stats.ttest_ind(defended, chased, equal_var=False)
# move 6 needs the effect size too
pooled = np.sqrt((defended.std()**2 + chased.std()**2) / 2)
d = (defended.mean() - chased.mean()) / pooled
print(d)
print(pooled)
print(t)
print(p)

0.9348281832945031
30.07625270385799
15.077334905536246
1.5743108504744986e-46


In [16]:
#Is the powerplay really faster than the middle overs?

#claim :Teams score faster in the first 
# six overs than they do in overs 7 to 15.
#h0= Both phases score at the same rate.
# Thegap we saw is luck.
#h1=The two run rates are not equal.



df=pd.read_sql("""SELECT match_id, innings,
       6.0 * SUM(CASE WHEN phase='Powerplay' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) AS pp,

       6.0 * SUM(CASE WHEN phase='Middle' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) AS mid

FROM v_ball

WHERE innings IN (1,2)

GROUP BY match_id, innings

HAVING
    SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) > 0
    AND
    SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) > 0;""",con)
df

,match_id,innings,pp,mid
0,335982,1,10.166667,10.333333
1,335982,2,4.333333,6.111111
2,335983,1,8.833333,12.000000
3,335983,2,10.500000,11.333333
4,335984,1,6.666667,6.222222
...,...,...,...,...
2401,1529284,2,15.333333,11.222222
2402,1529285,1,9.833333,8.111111
2403,1529285,2,11.500000,8.444444
2404,1529286,1,9.333333,10.111111


In [17]:
t,p= stats.ttest_rel(df.pp, df.mid)
print(round(t,2),round(p,4))

1.53 0.1259


In [18]:
t,p= stats.ttest_rel(df.pp, df.mid)
print(round(t,2))
print(round(p,4))


print(p<0.05)

1.53
0.1259
False


In [19]:
#Do left-handers really hit harder than right-handers?
#claim:Left-handed batters have a higher strike rate than right-handed batters.
#h0=Both hit at the same rate. The gap found is luck.
# h1=The two average strike rates are not equal.

df=pd.read_sql("""
SELECT batter,batsman_type,
       100.0*SUM(batter_runs)/SUM(is_legal) AS sr,
       SUM(is_legal) AS balls
FROM   v_ball
where  batsman_type IS NOT NULL
GROUP BY batter,batsman_type
HAVING balls >= 500;""",con)
df




,batter,batsman_type,sr,balls
0,A Badoni,Right hand Bat,139.263804,815
1,A Symonds,Right hand Bat,130.738255,745
2,AB de Villiers,Right hand Bat,152.181604,3392
3,AC Gilchrist,Left hand Bat,139.514498,1483
4,AD Mathews,Right hand Bat,126.352531,573
...,...,...,...,...
132,Washington Sundar,Left hand Bat,130.518234,521
133,Y Venugopal Rao,Right hand Bat,118.389423,832
134,YBK Jaiswal,Left hand Bat,154.104478,1608
135,YK Pathan,Right hand Bat,143.741588,2229


In [ ]:
# Select left-handed and right-handed batters
left = df[df.batsman_type == 'Left hand Bat']
right = df[df.batsman_type == 'Right hand Bat']

# Perform two-sample independent t-test (Welch's t-test)
t, p = stats.ttest_ind(left.sr, right.sr, equal_var=False)

print(t)
print(p)
print(p < 0.05)

-0.007335404976981389
0.994160816878741
False


In [ ]:

# Are spinners genuinely cheaper than fast bowlers?
# CLAIM: Spin bowlers concede fewer runs per over than fast bowlers
# h0 = Both styles concede the same. The gap found is luck.
# h1 = The two average economy rates are not equal

df = pd.read_sql("""
    SELECT bowler,
           CASE WHEN bowler_type LIKE '%Legbreak%'
                  OR bowler_type LIKE '%Offbreak%'
                  OR bowler_type LIKE '%Orthodox%'
                  OR bowler_type LIKE '%Googly%'
                  OR bowler_type LIKE '%Wrist spin%'
                THEN 'Spin' ELSE 'Pace' END AS style,
           6.0 * SUM(bowler_runs) / SUM(is_legal) AS economy,
           SUM(is_legal) AS balls
    FROM v_ball
    WHERE TRIM(COALESCE(bowler_type, '')) <> ''
    GROUP BY bowler, style
    HAVING balls >= 500;
""", con)

df




,bowler,style,economy,balls
0,A Kumble,Spin,6.578238,965
1,A Mishra,Spin,7.377633,3371
2,A Nehra,Pace,7.845912,1908
3,A Nortje,Pace,9.085714,1120
4,A Symonds,Spin,7.673624,527
...,...,...,...,...
144,YK Pathan,Spin,7.401918,1147
145,YS Chahal,Spin,8.027912,3941
146,Yash Dayal,Pace,9.575201,871
147,Yuvraj Singh,Spin,7.436133,869


In [22]:
spin = df[df['style'] == 'Spin']
pace = df[df['style'] == 'Pace']
t, p = stats.ttest_ind(spin.economy, pace.economy, equal_var=False)

print(t)
print(p)
print(p < 0.05)

-7.075597311698715
1.1249651433007181e-10
True
